# Colliders - Trail Difficulty, Speed and Accidents

**The setup.** `X` = trail difficulty (0 to 1), `Y` = speed (mph), `Z` = likelihood of an accident
(0 to 1). The causal structure is

$$X \longrightarrow Y, \qquad X \longrightarrow Z, \qquad Y \longrightarrow Z$$

Difficulty slows riders down, and both difficulty and speed push up the accident risk. `Z` is
therefore a **collider**: two arrows point into it.

**The experiment.** Rather than one data set, the data-generating code is run many times and a
regression is fit each time, so the question "what is the average coefficient if you do this
enough times?" is answered by simulation rather than by a single noisy estimate.

| # | Question | Where |
|---|---|---|
| 1 | Average coefficient of `X` when regressing `Y` on `X` alone | Section 3 |
| 2 | Average coefficient of `X` when regressing `Y` on `X` and `Z` | Section 4 |
| - | Should `Z` be controlled for? | Section 6 |

**What to expect.** The generating code sets speed to `normal(15, 5) - difficulty * 10`, floored
at zero, so the causal effect of difficulty on speed is about **-10** mph per unit of difficulty.
Regressing `Y` on `X` alone should land near that. Adding `Z` conditions on a collider, which
opens a path between `X` and `Y` that does not exist in the data-generating process, so that
coefficient should drift away from the truth.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NUM    = 100_000     # sample size inside each data set, as in the question's code
TRIALS = 100         # how many times the whole experiment is repeated
SEED   = 2024

pd.set_option("display.float_format", lambda v: f"{v:,.6f}")
rng = np.random.default_rng(SEED)

## 2. The data-generating process

This is the code from the question, wrapped in a function so it can be called repeatedly with a
fresh draw each time. `np.random.default_rng` replaces the legacy `np.random` calls so the whole
notebook is reproducible; the distributions are identical.

In [ ]:
def make_data(num=NUM, generator=None):
    g = generator if generator is not None else rng

    difficulty = g.uniform(0, 1, (num,))
    speed = np.maximum(g.normal(15, 5, (num,)) - difficulty * 10, 0)
    accident = np.minimum(
        np.maximum(0.03 * speed + 0.4 * difficulty + g.normal(0, 0.3, (num,)), 0), 1)

    return pd.DataFrame({"difficulty": difficulty, "speed": speed, "accident": accident})

df = make_data()
print("X = difficulty, Y = speed, Z = accident\n")
print(df.describe())
print(f"\nshare of speeds floored at 0: {(df['speed'] == 0).mean():.4f}")
print(f"share of accident values clipped at 0 or 1: "
      f"{((df['accident'] == 0) | (df['accident'] == 1)).mean():.4f}")
df.head()

In [ ]:
def ols(y, *regressors):
    # Returns the coefficient vector for an intercept plus the regressors given.
    X = np.column_stack([np.ones(len(y))] + [np.asarray(r, float) for r in regressors])
    beta, *_ = np.linalg.lstsq(X, np.asarray(y, float), rcond=None)
    return beta

b = ols(df["speed"], df["difficulty"])
print(f"single data set, Y on X        : intercept {b[0]:.6f}, coefficient of X {b[1]:.6f}")
b2 = ols(df["speed"], df["difficulty"], df["accident"])
print(f"single data set, Y on X and Z  : intercept {b2[0]:.6f}, "
      f"coefficient of X {b2[1]:.6f}, coefficient of Z {b2[2]:.6f}")

## 3. Question 1 - regress `Y` on `X`, many times

Each trial draws a brand-new data set of `NUM` rows and fits
$$\text{speed} = \beta_0 + \beta_1\,\text{difficulty} + \varepsilon$$
The answer is the average of $\beta_1$ across the trials.

In [ ]:
sim_rng = np.random.default_rng(SEED + 1)

coef_x_only = np.empty(TRIALS)
coef_x_with_z = np.empty(TRIALS)
coef_z = np.empty(TRIALS)

for i in range(TRIALS):
    d = make_data(NUM, sim_rng)
    y = d["speed"].to_numpy()
    x = d["difficulty"].to_numpy()
    z = d["accident"].to_numpy()

    coef_x_only[i] = ols(y, x)[1]
    both = ols(y, x, z)
    coef_x_with_z[i] = both[1]
    coef_z[i] = both[2]

print(f"{TRIALS} trials of {NUM:,} rows each\n")
print(f">>> Q1  average coefficient of X (Y on X alone) = {coef_x_only.mean():.6f}")
print(f"        std dev across trials {coef_x_only.std(ddof=1):.6f}, "
      f"std error of the mean {coef_x_only.std(ddof=1) / np.sqrt(TRIALS):.6f}")
print(f"        range across trials: {coef_x_only.min():.6f} to {coef_x_only.max():.6f}")

## 4. Question 2 - regress `Y` on `X` and `Z`, many times

$$\text{speed} = \beta_0 + \beta_1\,\text{difficulty} + \beta_2\,\text{accident} + \varepsilon$$

Same trials, same data, one extra regressor.

In [ ]:
print(f">>> Q2  average coefficient of X (Y on X and Z) = {coef_x_with_z.mean():.6f}")
print(f"        std dev across trials {coef_x_with_z.std(ddof=1):.6f}, "
      f"std error of the mean {coef_x_with_z.std(ddof=1) / np.sqrt(TRIALS):.6f}")
print(f"        range across trials: {coef_x_with_z.min():.6f} to {coef_x_with_z.max():.6f}")
print(f"\n        average coefficient of Z for reference: {coef_z.mean():.6f}")

results = pd.DataFrame({
    "Y on X alone": [coef_x_only.mean(), coef_x_only.std(ddof=1),
                     coef_x_only.std(ddof=1) / np.sqrt(TRIALS)],
    "Y on X and Z": [coef_x_with_z.mean(), coef_x_with_z.std(ddof=1),
                     coef_x_with_z.std(ddof=1) / np.sqrt(TRIALS)],
}, index=["average coefficient of X", "std dev across trials", "std error of the mean"])
results

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.hist(coef_x_only, bins=25, alpha=0.75, color="steelblue", label="Y on X alone")
ax.hist(coef_x_with_z, bins=25, alpha=0.75, color="crimson", label="Y on X and Z")
ax.axvline(-10, color="black", ls="--", lw=2, label="causal effect in the code (-10)")
ax.axvline(coef_x_only.mean(), color="steelblue", lw=2)
ax.axvline(coef_x_with_z.mean(), color="crimson", lw=2)
ax.set_xlabel("estimated coefficient of X")
ax.set_ylabel(f"count out of {TRIALS} trials")
ax.set_title("Where each specification lands")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Why the first number is not exactly -10

The code writes `np.maximum(..., 0)`, so speeds that would have come out negative are set to
zero. That flattens the bottom of the relationship a little and pulls the slope slightly toward
zero. Re-running without the floor shows the size of that distortion, which is a property of the
data-generating code and not of the regression.

In [ ]:
check_rng = np.random.default_rng(SEED + 99)
raw_coefs, floored_coefs = [], []
for _ in range(20):
    difficulty = check_rng.uniform(0, 1, (NUM,))
    unfloored = check_rng.normal(15, 5, (NUM,)) - difficulty * 10
    floored = np.maximum(unfloored, 0)
    raw_coefs.append(ols(unfloored, difficulty)[1])
    floored_coefs.append(ols(floored, difficulty)[1])

print(f"slope with no floor applied : {np.mean(raw_coefs):.6f}   (the -10 written in the code)")
print(f"slope after the floor at 0  : {np.mean(floored_coefs):.6f}")
print(f"attenuation from the floor  : {np.mean(floored_coefs) - np.mean(raw_coefs):+.6f}")

## 6. Should `Z` be controlled for?

**No.** `Z` is a collider on the path $X \rightarrow Z \leftarrow Y$. Conditioning on a collider
creates an association between its two causes that is not there otherwise, so holding accident
risk fixed manufactures a relationship between difficulty and speed on top of the real one.

The intuition from the example: among riders who all share the same accident risk, a rider on a
harder trail must have been going *slower* than a rider on an easier trail, since difficulty and
speed are the two things that produce risk. That trade-off is forced by the conditioning, and the
regression reads it as part of the effect of difficulty on speed.

`X` is not confounded with `Y` here - nothing causes both of them - so the simple regression of
`Y` on `X` already recovers the causal effect. Adding `Z` can only damage it. The general rule:
control for confounders (common causes), never for colliders (common effects), and never for
anything downstream of the treatment.

The cell below makes the induced association visible by slicing the data into narrow bands of
`Z` and fitting the `X`-`Y` relationship inside each band.

In [ ]:
d = make_data(NUM, np.random.default_rng(SEED + 7))
d["z_band"] = pd.qcut(d["accident"], q=5)

rows = []
for band, g in d.groupby("z_band", observed=True):
    rows.append({"accident band": str(band), "n": len(g),
                 "slope of Y on X inside the band": ols(g["speed"], g["difficulty"])[1]})
within = pd.DataFrame(rows)

print(f"slope ignoring Z entirely: {ols(d['speed'], d['difficulty'])[1]:.6f}\n")
print("inside narrow bands of Z the slope is pulled away from the truth:")
within

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sample = d.sample(4000, random_state=0)

axes[0].scatter(sample["difficulty"], sample["speed"], s=5, alpha=0.25, color="grey")
grid = np.linspace(0, 1, 50)
bb = ols(d["speed"], d["difficulty"])
axes[0].plot(grid, bb[0] + bb[1] * grid, color="black", lw=2.5,
             label=f"ignoring Z: slope {bb[1]:.3f}")
axes[0].set_xlabel("X = difficulty"); axes[0].set_ylabel("Y = speed")
axes[0].set_title("All the data: the causal relationship")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

colors = plt.cm.viridis(np.linspace(0, 0.9, d["z_band"].nunique()))
for (band, g), col in zip(d.groupby("z_band", observed=True), colors):
    gs = g.sample(min(800, len(g)), random_state=0)
    axes[1].scatter(gs["difficulty"], gs["speed"], s=5, alpha=0.3, color=col)
    bg = ols(g["speed"], g["difficulty"])
    axes[1].plot(grid, bg[0] + bg[1] * grid, color=col, lw=2.5,
                 label=f"Z in {band}: slope {bg[1]:.2f}")
axes[1].set_xlabel("X = difficulty"); axes[1].set_ylabel("Y = speed")
axes[1].set_title("Sliced by Z: conditioning on the collider bends every slice")
axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Answers

In [ ]:
print(f"Monte Carlo over {TRIALS} data sets of {NUM:,} rows\n")
print(f"  Q1  Y regressed on X alone")
print(f"        average coefficient of X = {coef_x_only.mean():.4f}")
print(f"        (the causal effect written into the code is -10, slightly attenuated")
print(f"         by the floor at zero speed)\n")
print(f"  Q2  Y regressed on X and Z")
print(f"        average coefficient of X = {coef_x_with_z.mean():.4f}")
print(f"        shifted by {coef_x_with_z.mean() - coef_x_only.mean():+.4f} "
      f"relative to the correct specification\n")
print("  Should Z be controlled for?  No - Z is a collider (X and Y both cause Z).")
print("  Conditioning on it invents an association between X and Y. There is no confounder")
print("  here, so the simple regression of Y on X is already the right answer.")